# EEG capstone: How robust is the eyes-closed alpha comparison?

Reproduce the first browser lab on the same two public recordings. Change the selected time window, then report what the result supports for this participant.

Run the cells in order in Jupyter or Colab. This notebook uses Python 3, MNE-Python, NumPy, and Matplotlib. In a fresh notebook environment, install them with `%pip install mne matplotlib`. The first run downloads about 2.5 MB of public EDF files.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import mne

BASE = "https://physionet.org/files/eegmmidb/1.0.0/S001/"
runs = {"Eyes open": "S001R01.edf", "Eyes closed": "S001R02.edf"}
data_dir = Path("eeg-data")
data_dir.mkdir(exist_ok=True)

for filename in runs.values():
    target = data_dir / filename
    if not target.exists():
        temporary = target.with_suffix(".edf.part")
        urlretrieve(BASE + filename, temporary)
        temporary.replace(target)

## 1. Inspect the excerpts

Fp1 is a frontal channel and O1 is an occipital channel. EDF channel labels in this dataset include trailing dots. Subtract the mean of each channel within the displayed segment to match the browser view. The recordings were collected at 160 Hz.

In [ ]:
def excerpt(condition, start):
    raw = mne.io.read_raw_edf(data_dir / runs[condition], preload=True, verbose="ERROR")
    return raw.pick(["Fp1.", "O1.."]).crop(start, start + 4, include_tmax=False)


fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
for condition, axis in zip(runs, axes):
    segment = excerpt(condition, 18)
    data_uv = segment.get_data(units="uV")
    centered = data_uv - data_uv.mean(axis=1, keepdims=True)
    for label, samples in zip(["Fp1", "O1"], centered):
        axis.plot(segment.times, samples, label=label)
    axis.set(title=condition, ylabel="µV")
    axis.legend(loc="upper right")
axes[-1].set_xlabel("Time in excerpt (s)")
plt.show()

## 2. Measure relative alpha power

Estimate the spectrum of O1 over one four-second segment using a Hann window. Divide summed power in 8–12 Hz by summed power in 4–30 Hz. The browser lab uses the same bands and a symmetric Hann window. MNE’s window convention can produce small differences.

In [ ]:
def alpha_share(condition, start):
    segment = excerpt(condition, start).pick(["O1.."])
    spectrum = segment.compute_psd(
        method="welch", fmin=4, fmax=30, n_fft=segment.n_times,
        n_per_seg=segment.n_times, n_overlap=0, window="hann",
        remove_dc=True, verbose="ERROR",
    )
    power, frequency = spectrum.get_data(return_freqs=True)
    alpha = (frequency >= 8) & (frequency <= 12)
    return power[0, alpha].sum() / power[0].sum()


for condition in runs:
    print(f"{condition}: {alpha_share(condition, 18):.0%}")

## 3. Test another time window

Compare several start times to see whether the size of the contrast depends on where you look. These excerpts all come from the same participant.

In [ ]:
for start in [2, 10, 18, 26]:
    opened = alpha_share("Eyes open", start)
    closed = alpha_share("Eyes closed", start)
    print(f"{start:2d}–{start + 4:2d} s   open {opened:5.0%}   closed {closed:5.0%}   gap {closed - opened:+.0%}")

## 4. Write a three-sentence result

1. Report both alpha shares at 18 seconds and identify which channel you measured.
2. Say whether the direction or size of the contrast changed across excerpts.
3. Describe one follow-up you would need before generalizing to other people.

At 18 seconds you should get about 31% for eyes open and 71% for eyes closed. These percentages are ratios of spectral power in one channel and band. They do not measure the percentage of brain activity or a person’s level of focus.

## Data and credit

Gerwin Schalk and colleagues, Wadsworth Center and BCI2000. [EEG Motor Movement/Imagery Dataset v1.0.0](https://physionet.org/content/eegmmidb/1.0.0/) (Schalk, 2009, DOI 10.13026/C28G6P), distributed by PhysioNet under the [Open Data Commons Attribution License v1.0](https://opendatacommons.org/licenses/by/1-0/). See the dataset page for citations to the original BCI2000 paper and PhysioNet.